In [1]:
import json
import os
from langchain.llms import Ollama
from langchain.text_splitter import CharacterTextSplitter
import openai
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI

In [2]:

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 


llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)


In [3]:
# Initialize OpenAI client (new method)
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def ask_openai(question, model="gpt-4o"):
    """Sends a question to OpenAI's API and returns the response."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# Example Usage
question = "What is differential privacy?"
answer = ask_openai(question)
print("OpenAI Response:", answer)


OpenAI Response: Differential privacy is a mathematical framework and approach used to ensure that the privacy of individuals is protected when their data is included in aggregate datasets and analyses. The main idea behind differential privacy is to allow analysts to extract useful statistical information from a large dataset while providing strong guarantees that the removal or addition of a single individual's data does not significantly affect the outcome of any analysis.

Here are some key aspects of differential privacy:

1. **Privacy Guarantee**: Differential privacy makes it difficult for an adversary to determine whether a particular individual's data is included in the dataset. This is done by ensuring that the output of any data analysis is statistically indistinguishable when any single individual's data changes.

2. **Mathematical Definition**: It is mathematically defined using parameters (typically ε, known as the privacy loss parameter, and sometimes δ). A system is con

In [4]:
ollama = Ollama(base_url='http://localhost:11434', model="llama3.1:70b")

/tmp/ipykernel_4127563/2524497889.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434', model="llama3.1:70b")


In [5]:
# system_prompt = """
# # Hybrid Knowledge Graph and Keyword Extraction Agent

# ## Role
# You are an advanced information extraction agent specialized in building comprehensive knowledge graphs and keyword indexes from unstructured text. Your output is used for semantic search, indexing, reasoning, and data validation.

# ## Objective
# From the provided text, extract:
# 1. **Entities as nodes** with relevant attributes.
# 2. **Relationships between entities** with relevant attributes.
# 3. **High-quality keywords as nodes**, directly linked to the document for fast indexing.
# 4. **The document node must store the content of the document itself.**

# All outputs must be included in a **single, valid JSON object** following the defined structure.

# ## Allowed Labels
# Each keyword node must use one label from the following list:

# { "person", "organization", "location", "event", "date", "work", "law", "product", "language", "scientific_term", "other" }

# If a term doesn't fit any category clearly, use `"other"`.

# ---

# ## Output Format (JSON)
# ```json
# {
#   "nodes": [
#     {
#       "id": "unique_node_id",
#       "label": "nodetype",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "id": "unique_keyword_id",
#       "label": "keyword_label"
#     },
#     {
#       "id": "docX",
#       "label": "document",
#       "attributes": {
#         "content": "Original document content here."
#       }
#     }
#   ],
#   "relationships": [
#     {
#       "source": "source_node_id",
#       "target": "target_node_id",
#       "type": "RELATIONSHIP_TYPE",
#       "attributes": {
#         "key1": "value1"
#       }
#     },
#     {
#       "source": "unique_keyword_id",
#       "target": "docX",
#       "type": "MENTIONED_IN"
#     }
#   ]
# }

# """


In [6]:

system_prompt = """
# Hybrid Knowledge Graph and Keyword Extraction Agent

## Role  
You are an advanced information extraction agent specialized in building comprehensive knowledge graphs and keyword indexes from unstructured text. Your output is used for semantic search, indexing, reasoning, and data validation.

---

## Objective  
From the provided text, extract:  
1. **Entities as nodes** with relevant attributes.  
2. **Relationships between entities** with relevant attributes.  
3. **High-quality keywords as nodes**, directly linked to the document for fast indexing.  
4. **The document node must store the content of the document itself.**  
5. **All extracted nodes (entities and keywords) must have a MENTIONED_IN relationship pointing to the document node.**  

Return results as a **single, valid JSON object** following the defined structure.

---

## Allowed Labels  
Each node must use one label from the following list:  

{ "person", "organization", "location", "event", "date", "work", "law", "product", "language", "scientific_term", "other" }  

If a term doesn't clearly fit any category, use `"other"`.

---

## Output Format (JSON)  

{
  "nodes": [
    {
      "id": "unique_node_id",
      "label": "nodetype",
      "attributes": {
        "key1": "value1",
        "key2": "value2"
      }
    },
    {
      "id": "unique_keyword_id",
      "label": "keyword_label"
    },
    {
      "id": "docX",
      "label": "document",
      "attributes": {
        "content": "Original document content here."
      }
    }
  ],
  "relationships": [
    {
      "source": "source_node_id",
      "target": "target_node_id",
      "type": "RELATIONSHIP_TYPE",
      "attributes": {
        "key1": "value1"
      }
    },
    {
      "source": "unique_node_or_keyword_id",
      "target": "docX",
      "type": "MENTIONED_IN"
    }
  ]
}

Extraction Guidelines
1. Entity Nodes
Extract: persons, organizations, locations, events, products, laws, scientific terms, works, concepts.

Use camelCase for attributes (e.g., name, role, episodeCount).

IDs: lowercase, singular, underscores for spaces.

Assign correct labels based on real-world categories.

2. Label Assignment
Example Entity	Label
Person name	person
Company, media	organization
City, country	location
Historical event	event
Specific date	date
Movie, TV show	entertainment
Law, policy	law
Product	product
Language	language
Scientific term	scientific_term
Unclear cases	other

Correct Examples:

"chicago_fire_season_4" → entertainment

"Srushti Bhavsar" → work

3. Keyword Nodes
Extract important noun phrases and domain terms.

Exclude stopwords, verbs, vague terms.

IDs: lowercase, singular, underscores for spaces.

Assign appropriate label or use "other" if unclear.

4. Document Node
Always include with id: "docX", label: "document", and content in attributes. also docX is just example. DO NOT use it as it is.

5. Relationships
Use ALL CAPS for relationship types (e.g., PRODUCED, LOCATED_IN).

Include attributes with camelCase keys.

MANDATORY: Every node must have a MENTIONED_IN link to the document node.

Do not add attributes to MENTIONED_IN.

6. Numbers and Dates
Dates: Use "YYYY-MM-DD" ISO format, store as attributes (e.g., releaseDate), not nodes.

Numbers: Store with descriptive keys (e.g., "episodeCount": 23), large numbers as strings. Use qualitative descriptors if unknown (e.g., "ownershipPercentage": "less than 50"). No separate number nodes.

7. IDs
Lowercase, underscore-separated, unique. No spaces or special characters.

8. Strict Compliance
Output valid JSON only, no markdown or comments.

If no data is extracted, return:

{ "nodes": [], "relationships": [] }

"""

In [7]:
# Function to read text from a file
def read_text_from_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


In [8]:
def extract_knowledge_graph(input_text_chunk, doc_id, labels_list):
    prompt = f"{system_prompt}\n\nDocument ID:\n{doc_id}\n\nInput Text:\n{input_text_chunk}"
    response = ask_openai(prompt)
    # Clean up markdown code block markers if present
    if response.strip().startswith("```"):
        response = response.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    
    return response


In [9]:
def process_single_document(doc, combined_graph, labels_list, failed_chunks_file, max_retries=2):
    doc_id = doc["_id"]
    text = doc["text"]
    retries = 0

    while retries < max_retries:
        try:
            response = extract_knowledge_graph(text, doc_id, list(labels_list))
            extracted_graph = json.loads(response)

            for node in extracted_graph.get("nodes", []):
                if "label" in node:
                    labels_list.add(node["label"])
                if node not in combined_graph["nodes"]:
                    combined_graph["nodes"].append(node)

            for rel in extracted_graph.get("relationships", []):
                if rel not in combined_graph["relationships"]:
                    combined_graph["relationships"].append(rel)


            break  # success
        except json.JSONDecodeError as e:
            retries += 1
            print(f"[ERROR] JSONDecodeError on doc {doc_id}, retry {retries}/{max_retries}: {e}")
            if retries >= max_retries:
                with open(failed_chunks_file, "r+", encoding="utf-8") as f:
                    failed_responses = json.load(f)
                    failed_responses.append({
                        "doc_id": doc_id,
                        "text": text,
                        "response": response
                    })
                    f.seek(0)
                    json.dump(failed_responses, f, indent=4)


In [10]:
def process_jsonl_file(input_path, output_path, failed_chunks_file):
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            combined_graph = json.load(f)
    else:
        combined_graph = {"nodes": [], "relationships": []}

    if not os.path.exists(failed_chunks_file):
        with open(failed_chunks_file, "w", encoding="utf-8") as f:
            json.dump([], f, indent=4)

    labels_list = {node["label"] for node in combined_graph["nodes"] if "label" in node}

    with open(input_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                doc = json.loads(line)
                print(f"\nProcessing document {line_num}: {doc.get('_id')}")
                process_single_document(doc, combined_graph, labels_list, failed_chunks_file)
                with open(output_path, "w", encoding="utf-8") as out_f:
                    json.dump(combined_graph, out_f, indent=4)
            except Exception as e:
                print(f"[ERROR] Failed to process document {line_num}: {e}")

    print(f"\nCompleted. Graph saved at {output_path}.")

In [ ]:
# File paths
input_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/datasets/nq/hybrid.jsonl"
output_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/input_jsons/28_32k.json"
failed_chunks_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/input_jsons/failed_28_32k.json"

# Run the processor
process_jsonl_file(input_file_path, output_file_path, failed_chunks_file_path)


Processing document 1: doc28001
